# Sea-Surface Temperature (SST) Downscaling: Multi-Model Comparative Inference

This interactive notebook runs side-by-side inference across **all 4 deep learning downscaling paradigms** trained on the Australasian OFAM 10km grid (16x super-resolution: 32×32 $\to$ 512×512):

| Model | Paradigm | Parameter Count | Checkpoint Path |
|---|---|---|---|
| **SRDCNN** | Conventional Transpose-Convolution Baseline | **608,705 (~0.61M)** | `runs/srdn_srdcnn_mask_aware_f16` |
| **ResAFNO** | Adaptive Fourier Neural Operator + Progressive Upsampling | **4,832,721 (~4.83M)** | `runs/srdn_resafno_mask_aware_f16` |
| **Conditional GAN** | Residual-in-Residual Dense Blocks (RRDB) + PatchCritic | **~4,815,000 (~4.8M)** | `runs/gan_sr_v3_hard_consistency` |
| **Flow Matching** | Continuous-Time OT-CFM Velocity U-Net | **~4,920,000 (~4.9M)** | `runs/flow_sr_continue_220k` |


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('/esi/project/niwa03712/rampaln/PUBLICATIONS/2026/SSTDownscaling')
sys.path.insert(0, str(PROJECT_ROOT / 'SRDN'))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print("Project root:", PROJECT_ROOT)
print("Python executable:", sys.executable)

In [ ]:
# Load test dataset sample
data_candidates = [
    PROJECT_ROOT / "sst_10km_OFAM_historical_Australia.nc",
    Path("/g/data/sd82/sst_stand_10km_OFAM_historical_Australia_lon_interp.nc")
]
data_file = None
for p in data_candidates:
    if p.exists():
        data_file = p
        break

ds = xr.open_dataset(data_file, decode_times=False)
sst_var = ds["temp"]
if "st_ocean" in sst_var.dims and sst_var.sizes["st_ocean"] == 1:
    sst_var = sst_var.squeeze("st_ocean")

print("SST dataset opened successfully:", data_file)
print("Grid dimensions (time, lat, lon):", sst_var.shape)

# Select a test sample index from the holdout period (e.g., sample 12000 in 2012)
sample_idx = 12000
raw_target = np.asarray(sst_var[sample_idx, :, :].values, dtype=np.float32)

# Standardization parameters (OFAM Australasia)
mean_val = float(ds["temp_mean"].values) if "temp_mean" in ds else 20.658
std_val = float(ds["temp_std"].values) if "temp_std" in ds else 8.520

# Ocean mask (True for valid ocean cells, False for land)
ocean_mask = np.isfinite(raw_target) & (np.abs(raw_target) > 1e-5)

# Normalized high-res field (target)
target_norm = np.where(ocean_mask, (raw_target - mean_val) / std_val, 0.0).astype(np.float32)

# Create 16x low-res input (32x32) using 16x16 block averaging
shrink = 16
h, w = target_norm.shape
coarse_h, coarse_w = h // shrink, w // shrink
reshaped = target_norm.reshape(coarse_h, shrink, coarse_w, shrink)
reshaped_mask = ocean_mask.reshape(coarse_h, shrink, coarse_w, shrink)
block_counts = reshaped_mask.sum(axis=(1, 3))
coarse_norm = np.zeros((coarse_h, coarse_w), dtype=np.float32)
valid_coarse = block_counts > 0
coarse_norm[valid_coarse] = reshaped.sum(axis=(1, 3))[valid_coarse] / block_counts[valid_coarse]

print(f"Coarse predictor shape: {coarse_norm.shape} -> Target shape: {target_norm.shape}")

In [ ]:
predictions = {}
param_counts = {}

# -----------------------------------------------------
# 1. Baseline Bilinear Interpolation
# -----------------------------------------------------
import scipy.ndimage
coarse_bilinear = scipy.ndimage.zoom(coarse_norm, zoom=16.0, order=1)
predictions["Bilinear"] = np.where(ocean_mask, coarse_bilinear * std_val + mean_val, np.nan)
param_counts["Bilinear"] = 0

# -----------------------------------------------------
# 2. TensorFlow Models: SRDCNN & ResAFNO
# -----------------------------------------------------
import tensorflow as tf
from model_srdn_advanced import SRDCNN_SST_v3, SRDN_ResAFNO_v4

# Prepare TF inputs dict
tf_coarse_sst = coarse_norm[None, :, :, None]
tf_coarse_mask = valid_coarse[None, :, :, None].astype(np.float32)
tf_fine_mask = ocean_mask[None, :, :, None].astype(np.float32)
tf_inputs = [tf_coarse_sst, tf_coarse_mask, tf_fine_mask]

# A. SRDCNN Baseline
srdcnn_dir = PROJECT_ROOT / "runs" / "srdn_srdcnn_mask_aware_f16"
srdcnn_weights = srdcnn_dir / "model.weights.h5"
srdcnn_model = SRDCNN_SST_v3(numHiddenUnits=64, shrink=16)
param_counts["SRDCNN"] = srdcnn_model.count_params()
if srdcnn_weights.exists():
    srdcnn_model.load_weights(str(srdcnn_weights))
    print(f"Loaded SRDCNN weights from {srdcnn_weights} ({param_counts['SRDCNN']:,} params)")
srdcnn_out = srdcnn_model(tf_inputs, training=False).numpy()[0, :, :, 0]
predictions["SRDCNN"] = np.where(ocean_mask, srdcnn_out * std_val + mean_val, np.nan)

# B. ResAFNO
resafno_dir = PROJECT_ROOT / "runs" / "srdn_resafno_mask_aware_f16"
resafno_weights = resafno_dir / "model.weights.h5"
resafno_model = SRDN_ResAFNO_v4(numHiddenUnits=128, trunk_blocks=6, num_freq_blocks=8, shrink=16)
param_counts["ResAFNO"] = resafno_model.count_params()
if resafno_weights.exists():
    resafno_model.load_weights(str(resafno_weights))
    print(f"Loaded ResAFNO weights from {resafno_weights} ({param_counts['ResAFNO']:,} params)")
resafno_out = resafno_model(tf_inputs, training=False).numpy()[0, :, :, 0]
predictions["ResAFNO"] = np.where(ocean_mask, resafno_out * std_val + mean_val, np.nan)

# -----------------------------------------------------
# 3. PyTorch Models: GAN & Flow Matching
# -----------------------------------------------------
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("PyTorch inference device:", device)

# A. Conditional GAN
from model_gan import Generator
gan_dir = PROJECT_ROOT / "runs" / "gan_sr_v3_hard_consistency"
gan_weights = gan_dir / "generator_ema.pt"
if not gan_weights.exists():
    gan_weights = gan_dir / "generator.pt"

gan_model = Generator(base_channels=48, rrdb_blocks=4, levels=4, enforce_coarse_consistency=True).to(device)
param_counts["GAN"] = sum(p.numel() for p in gan_model.parameters())
if gan_weights.exists():
    gan_state = torch.load(gan_weights, map_location=device)
    gan_model.load_state_dict(gan_state)
    print(f"Loaded GAN weights from {gan_weights} ({param_counts['GAN']:,} params)")
gan_model.eval()

with torch.no_grad():
    t_coarse = torch.from_numpy(coarse_norm[None, None, :, :]).to(device)
    t_coarse_mask = torch.from_numpy(valid_coarse[None, None, :, :].astype(np.float32)).to(device)
    t_condition = torch.cat([t_coarse, t_coarse_mask], dim=1)
    t_ocean_mask = torch.from_numpy(ocean_mask[None, None, :, :].astype(np.float32)).to(device)
    gan_pred = gan_model(t_condition, t_ocean_mask).cpu().numpy()[0, 0]
predictions["GAN"] = np.where(ocean_mask, gan_pred * std_val + mean_val, np.nan)

# B. Flow Matching (OT-CFM)
from model import CondUNet
from flow import ODCFMSolver
flow_dir = PROJECT_ROOT / "runs" / "flow_sr_continue_220k"
flow_weights = flow_dir / "model_ema.pt"
if not flow_weights.exists():
    flow_weights = flow_dir / "model.pt"

flow_model = CondUNet(in_channels=1, out_channels=1, condition_channels=2, base_channels=64).to(device)
param_counts["Flow"] = sum(p.numel() for p in flow_model.parameters())
if flow_weights.exists():
    flow_state = torch.load(flow_weights, map_location=device)
    flow_model.load_state_dict(flow_state)
    print(f"Loaded Flow Matching weights from {flow_weights} ({param_counts['Flow']:,} params)")
flow_model.eval()

with torch.no_grad():
    solver = ODCFMSolver(flow_model)
    flow_pred = solver.sample(t_condition, t_ocean_mask, steps=20).cpu().numpy()[0, 0]
predictions["Flow"] = np.where(ocean_mask, flow_pred * std_val + mean_val, np.nan)

target_sst = np.where(ocean_mask, raw_target, np.nan)
print("\nAll 4 downscaling models and references generated predictions successfully!")

In [ ]:
# Multi-panel comparative visualization
model_keys = ["Bilinear", "SRDCNN", "ResAFNO", "GAN", "Flow"]
fig, axes = plt.subplots(2, 3, figsize=(20, 12), constrained_layout=True)
axes = axes.ravel()

vmin, vmax = np.nanquantile(target_sst, [0.01, 0.99])

# Panel 0: Ground Truth Target
im0 = axes[0].imshow(np.ma.masked_invalid(target_sst), origin="lower", cmap="turbo", vmin=vmin, vmax=vmax)
axes[0].set_title("High-Resolution Target (OFAM Ground Truth)", fontsize=13, fontweight="bold")
plt.colorbar(im0, ax=axes[0], label="SST (°C)", shrink=0.8)

# Panels 1 to 5: Model Downscaled Predictions
for i, key in enumerate(model_keys, start=1):
    pred = predictions[key]
    im = axes[i].imshow(np.ma.masked_invalid(pred), origin="lower", cmap="turbo", vmin=vmin, vmax=vmax)
    p_count = param_counts[key]
    title = f"{key} Downscaling ({p_count/1e6:.2f}M params)" if p_count > 0 else f"{key} Downscaling"
    axes[i].set_title(title, fontsize=13, fontweight="bold")
    plt.colorbar(im, ax=axes[i], label="SST (°C)", shrink=0.8)

for ax in axes:
    ax.set_xlabel("Longitude Grid Index")
    ax.set_ylabel("Latitude Grid Index")

plt.suptitle(f"SST 16x Super-Resolution Multi-Model Comparison (Sample Index {sample_idx})", fontsize=16, y=1.02)
plt.show()

In [ ]:
# Error Maps (Prediction - Ground Truth)
fig, axes = plt.subplots(1, 5, figsize=(24, 4.5), constrained_layout=True)
err_limit = 1.5  # +/- 1.5 degC for clear error contrast

for ax, key in zip(axes, model_keys):
    err = predictions[key] - target_sst
    im = ax.imshow(np.ma.masked_invalid(err), origin="lower", cmap="RdBu_r", vmin=-err_limit, vmax=err_limit)
    rmse = np.sqrt(np.nanmean(err**2))
    mae = np.nanmean(np.abs(err))
    ax.set_title(f"{key} Difference\nRMSE: {rmse:.3f}°C | MAE: {mae:.3f}°C", fontsize=12)
    plt.colorbar(im, ax=ax, label="Error (°C)", shrink=0.8)

plt.suptitle("Spatial Error Comparison (Model − OFAM Ground Truth)", fontsize=15, y=1.05)
plt.show()

# Quantitative Comparison Table
import pandas as pd
metrics = []
for key in model_keys:
    err = predictions[key] - target_sst
    metrics.append({
        "Model": key,
        "Parameters": f"{param_counts[key]:,}",
        "RMSE (°C)": round(float(np.sqrt(np.nanmean(err**2))), 4),
        "MAE (°C)": round(float(np.nanmean(np.abs(err))), 4),
        "Max Abs Error (°C)": round(float(np.nanmax(np.abs(err))), 4),
        "Pred 99th % (°C)": round(float(np.nanquantile(predictions[key], 0.99)), 3),
    })
df_metrics = pd.DataFrame(metrics)
print("\n--- Multi-Model Quantitative Evaluation ---")
print(df_metrics.to_string(index=False))